# Automated Delta Lake Table Maintenance Pipeline - On-Demand

Automates enterprise Delta Lake table maintenance:
- **OPTIMIZE & Z-ORDER**: Compact small files and co-locate data by business keys for fast queries
- **VACUUM**: Remove uncommitted and stale transaction files older than retention threshold
- **ANALYZE TABLE**: Compute statistics to update the cost-based query optimizer (CBO)

Runs **on-demand** or as a scheduled Databricks workflow maintenance job across Bronze, Silver, and Gold layers.

In [ ]:
# Notebook Parameters
dbutils.widgets.text("config_dir", "", "Config Directory (Optional - Auto-resolved if blank)")
dbutils.widgets.dropdown("stage", "dev", ["dev", "test", "staging", "prod"], "Deployment Stage")
dbutils.widgets.dropdown("layer", "all", ["all", "bronze", "silver", "gold"], "Target Layer")
dbutils.widgets.text("table_name", "", "Single Table Override (e.g. catalog.schema.table)")
dbutils.widgets.text("client_id", "", "Client ID / Tenant (Optional)")
dbutils.widgets.text("vacuum_retain_hours", "168", "Vacuum Retention Hours (Default 168 = 7 days)")
dbutils.widgets.dropdown("dry_run", "false", ["false", "true"], "Dry Run (Audit Only)")

# Resolve config_dir and repo_root generically
import os
import sys
from pathlib import Path

raw_config_dir = ""
try:
    raw_config_dir = dbutils.widgets.get("config_dir").strip()
except Exception:
    raw_config_dir = ""

if raw_config_dir:
    config_dir = raw_config_dir
    repo_root = str(Path(config_dir).parent)
else:
    try:
        # Dynamically discover the notebook's workspace/repo location
        nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        repo_root = str(Path(nb_path).parent.parent)
        config_dir = str(Path(repo_root) / "configs")
    except Exception:
        cwd = Path(os.getcwd())
        if (cwd / "configs").exists():
            repo_root = str(cwd)
            config_dir = str(cwd / "configs")
        elif (cwd.parent / "configs").exists():
            repo_root = str(cwd.parent)
            config_dir = str(cwd.parent / "configs")
        else:
            repo_root = str(cwd)
            config_dir = "configs"

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

stage = dbutils.widgets.get("stage")
layer = dbutils.widgets.get("layer")
table_name = dbutils.widgets.get("table_name").strip() or None
client_id = dbutils.widgets.get("client_id").strip() or None
vacuum_retain_hours = int(dbutils.widgets.get("vacuum_retain_hours"))
dry_run = dbutils.widgets.get("dry_run").lower() == "true"

print(f"Executing Table Maintenance Pipeline:")
print(f"  - Stage:               {stage}")
print(f"  - Layer:               {layer}")
print(f"  - Single Table:        {table_name or 'N/A (All Layer Tables)'}")
print(f"  - Client Scope:        {client_id or 'Global'}")
print(f"  - Retention Hours:     {vacuum_retain_hours}")
print(f"  - Dry Run:             {dry_run}")

In [ ]:
import sys
from pathlib import Path
import json


from lakeforge.maintenance.table_maintenance import TableMaintenanceManager, create_maintenance_manager

mgr = create_maintenance_manager(spark=spark)

In [ ]:
if table_name:
    print(f"Executing maintenance on single target table: {table_name}")
    m_res = mgr.run_table_maintenance(
        table_name=table_name,
        vacuum_retain_hours=vacuum_retain_hours,
        dry_run=dry_run
    )
    print(json.dumps(m_res.__dict__, indent=2))
    dbutils.notebook.exit(json.dumps({"status": m_res.status, "table": table_name}))
else:
    print(f"Running layer-wide table maintenance for stage '{stage}' across layer '{layer}'...")
    summary = mgr.run_layer_maintenance(
        config_dir=config_dir,
        stage=stage,
        layer=layer,
        client_id=client_id,
        vacuum_retain_hours=vacuum_retain_hours,
        dry_run=dry_run
    )
    print("\n--- Maintenance Execution Summary ---")
    print(f"Tables Processed: {summary['tables_processed']}")
    for res in summary['results']:
        print(f"  • Table: {res['table_name']} | Status: {res['status']} | Ops: {res['operations_executed']}")
    
    dbutils.notebook.exit(json.dumps({
        "status": "SUCCESS",
        "tables_processed": summary['tables_processed'],
        "stage": stage,
        "layer": layer
    }))